# Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 1 2026–2027 Major Project**

## Objective
Analyze agricultural data across **Kharif, Rabi and Zaid** seasons to identify meaningful patterns, trends, relationships, variations and evidence-based recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
df.head()

## 1. Dataset overview

In [ ]:
print("Rows, Columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

### Data preparation
The dataset is inspected for dimensions, data types, missing values and duplicates. Calculations use available non-missing observations rather than inventing values.

In [ ]:
season_order = ["Kharif", "Rabi", "Zaid"]
df["Season"] = pd.Categorical(df["Season"], categories=season_order, ordered=True)

season_summary = df.groupby("Season", observed=True).agg(
    Farms=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Median_Yield=("Yield_Tonnes_Ha","median"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Water_Used=("Water_Used_m3","mean"),
    Avg_Risk=("Disease_Pest_Risk_pct","mean")
).round(2)
season_summary

## 2. Seasonal yield comparison

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(season_summary.index.astype(str), season_summary["Avg_Yield"])
plt.title("Average Yield by Season")
plt.xlabel("Season")
plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

**Finding:** The table and chart identify the season with the highest and lowest average yield using the dataset's computed values.

## 3. Profit comparison

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(season_summary.index.astype(str), season_summary["Avg_Profit"]/100000)
plt.title("Average Profit by Season")
plt.xlabel("Season")
plt.ylabel("Average Profit (₹ Lakh)")
plt.show()

**Finding:** Average profit is compared across Kharif, Rabi and Zaid. The computed table should be used to interpret the magnitude of the seasonal difference.

## 4. Water efficiency and water use

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(season_summary.index.astype(str), season_summary["Avg_Water_Efficiency"])
plt.title("Average Water Efficiency by Season")
plt.xlabel("Season")
plt.ylabel("Tonnes per 1,000 m³")
plt.show()

season_summary[["Avg_Water_Used","Avg_Water_Efficiency"]]

**Finding:** The table above compares water efficiency and water use across seasons, allowing the highest-demand and most efficient seasons to be identified directly from the computed values.

## 5. Yield distribution

In [ ]:
data=[df.loc[df["Season"]==s,"Yield_Tonnes_Ha"].dropna() for s in season_order]
plt.figure(figsize=(9,5))
plt.boxplot(data, labels=season_order)
plt.title("Yield Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

The box plot compares spread, central tendency and potential outliers in yield across the three seasons.

## 6. Irrigation method and season

In [ ]:
irrigation = df.groupby(["Season","Irrigation_Method"], observed=True).agg(
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean")
).reset_index()

print(irrigation.round(2))

pivot = irrigation.pivot(index="Irrigation_Method", columns="Season", values="Avg_Yield")
pivot.plot(kind="bar", figsize=(10,5))
plt.title("Average Yield by Irrigation Method and Season")
plt.xlabel("Irrigation Method")
plt.ylabel("Yield (Tonnes/Ha)")
plt.xticks(rotation=0)
plt.show()

**Finding:** The irrigation-season table above identifies the best-performing irrigation method within each season based on average yield.

## 7. Crop-level profitability

In [ ]:
crop_summary = df.groupby("Crop").agg(
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean")
).sort_values("Avg_Profit", ascending=False)

crop_summary.round(2)

In [ ]:
plt.figure(figsize=(10,5))
top = crop_summary.head(8)
plt.bar(top.index.astype(str), top["Avg_Profit"]/100000)
plt.title("Average Profit by Crop")
plt.xlabel("Crop")
plt.ylabel("Average Profit (₹ Lakh)")
plt.xticks(rotation=30, ha="right")
plt.show()

**Finding:** Crop-level profitability differs substantially. The ranked table and chart identify the highest-profit crops in this dataset.

## 8. Relationship checks

In [ ]:
numeric_cols = [
    "Rainfall_mm","Avg_Temperature_C","Humidity_pct","Sunlight_Hours_Day",
    "Soil_Moisture_pct","Nitrogen_kg_ha","Fertilizer_kg_ha",
    "Seed_Quality_Score","Yield_Tonnes_Ha","Profit_INR",
    "Water_Used_m3","Water_Efficiency_t_per_1000m3","Disease_Pest_Risk_pct"
]
df[numeric_cols].corr()["Yield_Tonnes_Ha"].sort_values(ascending=False).round(3)

Correlation measures association, not causation. Water-efficiency is mechanically related to yield/water use, so that relationship should be interpreted with particular care.

## Conclusions and recommendations

- Compare Kharif, Rabi and Zaid using yield, profit, water use and water efficiency together.
- Investigate seasons with lower yield or efficiency for resource-management opportunities.
- Select irrigation methods using season-specific performance rather than assuming one method is optimal everywhere.
- Consider crop profitability alongside water requirements, seasonal conditions and risk.
- Further work could include regional comparisons, statistical significance testing, predictive modeling and an interactive dashboard.